In [38]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

In [39]:
spark = SparkSession.builder.appName("ny_taxi_data").getOrCreate()
sc=spark.sparkContext

In [40]:
## Question 1
print(type(spark))
print("Spark version:", spark.version)
print("App name:", sc.appName)
print("Master:", sc.master)
print(spark.sparkContext.uiWebUrl)

<class 'pyspark.sql.session.SparkSession'>
Spark version: 4.1.1
App name: ny_taxi_data
Master: local[*]
http://mac:4042


In [7]:
!wget "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"

--2026-03-08 08:07:38--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2684:1400:b:20a5:b140:21, 2600:9000:2684:9400:b:20a5:b140:21, 2600:9000:2684:aa00:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2684:1400:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  11.6MB/s    in 5.8s    

2026-03-08 08:07:44 (11.6 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [41]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

In [42]:
df.show(20)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [43]:
schema_list = [(f.name, f.dataType.simpleString(), f.nullable) for f in df.schema.fields]
schema_list

[('VendorID', 'int', True),
 ('tpep_pickup_datetime', 'timestamp_ntz', True),
 ('tpep_dropoff_datetime', 'timestamp_ntz', True),
 ('passenger_count', 'bigint', True),
 ('trip_distance', 'double', True),
 ('RatecodeID', 'bigint', True),
 ('store_and_fwd_flag', 'string', True),
 ('PULocationID', 'int', True),
 ('DOLocationID', 'int', True),
 ('payment_type', 'bigint', True),
 ('fare_amount', 'double', True),
 ('extra', 'double', True),
 ('mta_tax', 'double', True),
 ('tip_amount', 'double', True),
 ('tolls_amount', 'double', True),
 ('improvement_surcharge', 'double', True),
 ('total_amount', 'double', True),
 ('congestion_surcharge', 'double', True),
 ('Airport_fee', 'double', True),
 ('cbd_congestion_fee', 'double', True)]

In [45]:
df_cast = (
    df
    # timestamps
    .withColumn("tpep_pickup_datetime",  F.col("tpep_pickup_datetime").cast("timestamp"))
    .withColumn("tpep_dropoff_datetime", F.col("tpep_dropoff_datetime").cast("timestamp"))

    # integer-ish columns (Spark often reads some as bigint)
    .withColumn("passenger_count", F.col("passenger_count").cast("int"))
    .withColumn("RatecodeID",      F.col("RatecodeID").cast("int"))
    .withColumn("payment_type",    F.col("payment_type").cast("int"))

    # (these are already int in your schema, but casting is harmless / explicit)
    .withColumn("VendorID",        F.col("VendorID").cast("int"))
    .withColumn("PULocationID",    F.col("PULocationID").cast("int"))
    .withColumn("DOLocationID",    F.col("DOLocationID").cast("int"))

    # keep money columns as double (already double in your schema)
    .withColumn("fare_amount",            F.col("fare_amount").cast("double"))
    .withColumn("extra",                  F.col("extra").cast("double"))
    .withColumn("mta_tax",                F.col("mta_tax").cast("double"))
    .withColumn("tip_amount",             F.col("tip_amount").cast("double"))
    .withColumn("tolls_amount",           F.col("tolls_amount").cast("double"))
    .withColumn("improvement_surcharge",  F.col("improvement_surcharge").cast("double"))
    .withColumn("total_amount",           F.col("total_amount").cast("double"))
    .withColumn("congestion_surcharge",   F.col("congestion_surcharge").cast("double"))
    .withColumn("Airport_fee",            F.col("Airport_fee").cast("double"))
    .withColumn("cbd_congestion_fee",     F.col("cbd_congestion_fee").cast("double"))
)

df_cast.printSchema()
df_cast.show(5)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-----

In [9]:
df4 = df.repartition(4)

In [10]:
df4.write.mode("overwrite").parquet("yellow_tripdata/2025/11")

In [46]:
## Question 2:
from pathlib import Path

p = Path("yellow_tripdata/2025/11/part-00000-e14665f6-b4a5-44aa-a5c9-ca943f9bcdfc-c000.snappy.parquet")
size_mb = p.stat().st_size / 1000000

print(f"{size_mb} MB")

25.573525 MB


In [47]:
## Question 3
mask = (df4["tpep_pickup_datetime"] >= "2025-11-15") & (df4["tpep_pickup_datetime"] < "2025-11-16")
print(f"{df4[mask].count()} trips")

162604 trips


In [48]:
## Question 4
df4_tripmax = (
    df4
    .filter(F.col("tpep_pickup_datetime").isNotNull() & F.col("tpep_dropoff_datetime").isNotNull())
    # convert timestamp_ntz -> timestamp
    .withColumn("pickup_ts",  F.col("tpep_pickup_datetime").cast("timestamp"))
    .withColumn("dropoff_ts", F.col("tpep_dropoff_datetime").cast("timestamp"))
    # now epoch seconds works
    .withColumn("journey_seconds", F.col("dropoff_ts").cast("long") - F.col("pickup_ts").cast("long"))
    .filter(F.col("journey_seconds") >= 0)
)

df4_tripmax.agg((F.max("journey_seconds")/F.lit(3600)).alias("max_hours")).show(truncate=False)

+-----------------+
|max_hours        |
+-----------------+
|90.64666666666666|
+-----------------+



In [23]:
## Question 5
print(spark.sparkContext.uiWebUrl)

http://mac:4042


In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 17:01:56--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2684:3400:b:20a5:b140:21, 2600:9000:2684:5600:b:20a5:b140:21, 2600:9000:2684:9400:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2684:3400:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-09 17:01:57 (5.74 GB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [ ]:
df_zone = spark.read.csv("taxi_zone_lookup.csv", header=True)

In [49]:
schema_list = [(f.name, f.dataType.simpleString(), f.nullable) for f in df_zone.schema.fields]
schema_list

[('LocationID', 'string', True),
 ('Borough', 'string', True),
 ('Zone', 'string', True),
 ('service_zone', 'string', True)]

In [35]:
df_zone.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [51]:
df_join = df4.join(df_zone,df4.PULocationID == df_zone.LocationID,"left")
schema_list = [(f.name, f.dataType.simpleString(), f.nullable) for f in df_join.schema.fields]
schema_list

[('VendorID', 'int', True),
 ('tpep_pickup_datetime', 'timestamp_ntz', True),
 ('tpep_dropoff_datetime', 'timestamp_ntz', True),
 ('passenger_count', 'bigint', True),
 ('trip_distance', 'double', True),
 ('RatecodeID', 'bigint', True),
 ('store_and_fwd_flag', 'string', True),
 ('PULocationID', 'int', True),
 ('DOLocationID', 'int', True),
 ('payment_type', 'bigint', True),
 ('fare_amount', 'double', True),
 ('extra', 'double', True),
 ('mta_tax', 'double', True),
 ('tip_amount', 'double', True),
 ('tolls_amount', 'double', True),
 ('improvement_surcharge', 'double', True),
 ('total_amount', 'double', True),
 ('congestion_surcharge', 'double', True),
 ('Airport_fee', 'double', True),
 ('cbd_congestion_fee', 'double', True),
 ('LocationID', 'string', True),
 ('Borough', 'string', True),
 ('Zone', 'string', True),
 ('service_zone', 'string', True)]

In [69]:
## Question 6
df_join.groupBy("Zone").count().sort(2).show()

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Eltingville/Annad...|    1|
|Governor's Island...|    1|
|       Arden Heights|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
|   Rossville/Woodrow|    4|
|         Great Kills|    4|
| Green-Wood Cemetery|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
|New Dorp/Midland ...|   14|
|       West Brighton|   14|
|             Oakwood|   14|
|        Crotona Park|   14|
|       Willets Point|   15|
|Breezy Point/Fort...|   16|
|Saint George/New ...|   17|
|       Broad Channel|   18|
|     Mariners Harbor|   21|
|Heartland Village...|   22|
+--------------------+-----+
only showing top 20 rows
